In [1]:
using Ferrite
using ModularEIT
using Images
using IterativeSolvers
using LinearAlgebra
using Plots
using Distributions
using Statistics
using Lux
using JLD2


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [28]:

σ = 0.0 # how much noise is added to the operator (now a *relative* noise level, see the Λn cell — try e.g. 0.05-0.2 to actually see an effect; 0.0 means no noise)
# Steers prox operators:
ρ_obj = 0.0 # for prox operator

num_modes = 255

255

In [29]:
img = load("../reconstructions/1096.jpg")
img_f = img .|> Float64
img_f .-= 0.5
img_f .*= 2.0
img_f .= exp.(img_f) # Just an idea maybe EIT reconstruction is easier with this.

150×150 Matrix{Float64}:
 1.38471   1.39561   1.4066    1.4066    …  1.89499   1.88019   1.88019
 1.41768   1.42884   1.44009   1.44009      1.90992   1.90992   1.90992
 1.44009   1.45143   1.46286   1.46286      1.92495   1.92495   1.92495
 1.46286   1.47438   1.48599   1.48599      1.92495   1.92495   1.92495
 1.48599   1.49769   1.50948   1.50948      1.94011   1.94011   1.94011
 1.47438   1.48599   1.49769   1.49769   …  1.97078   1.97078   1.97078
 1.44009   1.45143   1.46286   1.46286      1.9863    1.9863    1.9863
 1.44009   1.45143   1.46286   1.46286      1.97078   1.9863    1.9863
 1.46286   1.46286   1.47438   1.48599      2.01771   2.00194   1.9863
 1.47438   1.48599   1.48599   1.49769      2.00194   2.00194   1.9863
 ⋮                                       ⋱                      
 0.367879  0.382593  0.388641  0.407368     0.388641  0.373696  0.388641
 0.394786  0.437162  0.385605  0.385605     0.394786  0.367879  0.373696
 0.385605  0.417066  0.42366   0.417066     0.39

In [30]:
itp = interpolate_array_2D(Float64.(img_f))

#interpolate_array_2D##0 (generic function with 1 method)

In [31]:
n = 63 
grid = generate_grid(Quadrilateral, (n, n));

In [32]:
∂Ω = union(getfacetset.((grid,), ["left", "top", "right", "bottom"])...)
fe  = FerriteFESpace{RefQuadrilateral}(grid,2,0,3,∂Ω)

FerriteFESpace{RefQuadrilateral}(CellValues{Ferrite.FunctionValues{1, Lagrange{RefQuadrilateral, 2}, Matrix{Float64}, Matrix{Vec{2, Float64}}, Matrix{Vec{2, Float64}}, Nothing, Nothing}, Ferrite.GeometryMapping{1, Lagrange{RefQuadrilateral, 1}, Matrix{Float64}, Matrix{Vec{2, Float64}}, Nothing}, QuadratureRule{RefQuadrilateral, Vector{Float64}, Vector{Vec{2, Float64}}}, Vector{Float64}}(Ferrite.FunctionValues{1, Lagrange{RefQuadrilateral, 2}, Matrix{Float64}, Matrix{Vec{2, Float64}}, Matrix{Vec{2, Float64}}, Nothing, Nothing}(Lagrange{RefQuadrilateral, 2}(), [0.47237900077244527 -1.4605338811812958e-17 … 1.8551212633834227e-18 0.007620999227554982; -0.05999999999999999 1.4605338811812958e-17 … -1.8551212633834227e-18 -0.059999999999999984; … ; 0.2749193338482966 -8.50014503228635e-18 … -8.500145032286352e-18 -0.03491933384829665; 0.15999999999999984 0.3999999999999998 … 0.3999999999999999 0.15999999999999992], [0.47237900077244527 -1.4605338811812958e-17 … 1.8551212633834227e-18 0.0076

In [33]:
cond_vec = project_function_to_fem(fe, itp; space=:σ)
cond_vec .= min.(max.(cond_vec, 1e-6), 2.8)

3969-element SparseArrays.SparseVector{Float64, Int64} with 3969 stored entries:
  [1   ]  =  1.43257
  [2   ]  =  1.4845
  [3   ]  =  1.46419
  [4   ]  =  1.47182
  [5   ]  =  1.49761
  [6   ]  =  1.5537
  [7   ]  =  1.59288
          ⋮
  [3962]  =  0.391354
  [3963]  =  0.405863
  [3964]  =  0.443788
  [3965]  =  0.403251
  [3966]  =  0.383249
  [3967]  =  0.386418
  [3968]  =  0.377171
  [3969]  =  0.382861

In [34]:
eval_points = reshape(equidistant_grid(64), :)
ph = PointEvalHandler(grid, eval_points)

PointEvalHandler{Grid{2, Quadrilateral, Float64}, Float64}
  number of points: 4096
  Found corresponding cell for all points.

In [35]:
# G must be mean zero on the boundary (Neumann data integrates to 0);
# `down`/`up` batch over columns, so this normalizes all modes at once.
mean_zero_boundary(G_boundary) = G_boundary .- Statistics.mean(G_boundary, dims=1)

mean_zero_boundary (generic function with 1 method)

In [36]:
G_full = real_fourier_basis(8)
rhs_dict = Dict()
Threads.@threads for i in 2:num_modes+1
    M = make_boundary(G_full[:, i],64)
    itp = interpolate_array_2D(M)
    rhs_dict[i-1] = assemble_rhs_func(fe, itp)
end
G = reduce(hcat, [rhs_dict[i] for i in 1:num_modes])
G = fe.up(mean_zero_boundary(fe.down(G)))

16129×255 Matrix{Float64}:
  0.000935326   6.25191e-21  0.000935326  …  -1.74317e-19   0.000661344
  0.000935036   2.33183e-5   0.000934164      2.25898e-5   -0.000661407
  0.0           0.0          0.0              0.0           0.0
  0.000935036  -2.33183e-5   0.000934164     -2.25898e-5   -0.000661407
  0.00187037    2.33184e-5   0.00186951       2.33184e-5   -2.10274e-5
  0.0           0.0          0.0          …   0.0           0.0
  0.0           0.0          0.0              0.0           0.0
  0.00187037   -2.33184e-5   0.00186951      -2.33184e-5   -2.10274e-5
  0.0           0.0          0.0              0.0           0.0
  0.000934164   4.66221e-5   0.00093068      -4.51665e-5    0.000661344
  ⋮                                       ⋱                
  0.0           0.0          0.0          …   0.0           0.0
 -0.000935036  -2.33183e-5   0.000934164     -2.25898e-5   -0.000661407
  0.0           0.0          0.0              0.0           0.0
 -0.0018692    -6.99407e-5 

In [37]:
G_in = G
fbm_true = FerriteBlockMode(cond_vec, G_in, fe; block=true)

FerriteBlockMode(ModularEIT.BlockLAssembler(9, [0.23822472205239942 -0.012762038681378523 … -0.07657223208827142 -0.13612841260137062; -0.012762038681378523 0.23822472205240014 … 0.042540128937928745 -0.1361284126013715; … ; -0.07657223208827142 0.042540128937928745 … 0.7487062693075427 -0.40838523780411423; -0.13612841260137062 -0.1361284126013715 … -0.40838523780411423 2.178054601621942]), [1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0  …  1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0], sparse([1, 2, 3, 4, 5, 6, 7, 8, 9, 1  …  16129, 15868, 15872, 15874, 16122, 16123, 16126, 16127, 16128, 16129], [1, 1, 1, 1, 1, 1, 1, 1, 1, 2  …  16128, 16129, 16129, 16129, 16129, 16129, 16129, 16129, 16129, 16129], [0.8913780897139235, -0.04775239766324571, -0.031834931775497274, -0.04775239766324583, -0.28651438597947626, 0.15917465887748627, 0.15917465887748627, -0.286514385979475, -0.5093589084079558, -0.04775239766324571  …  -0.4083852378041141, -0.13612841260137062, -0.1361284126013715, -

In [38]:
F = copy(fbm_true.F)
F = mean_zero_boundary(F)
Λ = F * pinv(fe.down(G))
Λ = 0.5 .* (Λ' + Λ) # It should be self adjoint, so I make it self adjoint. (It basically is up to small numerical errors)
# This is supposed to be a synthetic EIT reconstruction, so I should somehow add noise to my reconstruction. I was thinking about adding symmetric random noise matrix to Λ or Λ⁻¹. Please do:
# NOTE: the line below was previously `Λn = Λ + σ * ` — incomplete syntax (nothing after `*`), which is why this cell couldn't run.
# Two things worth fixing beyond that:
#   1. Λ is symmetric (you just enforced that above), so the noise should be symmetric too —
#      an unsymmetrized `randn` perturbation adds a skew component that isn't physical and will
#      show up as spurious asymmetry in Λn (and later mess with svd_on_modes/reconstruction).
#   2. σ should act as a *relative* noise level (this matches how σ is described in the parameter
#      cell, and is the usual EIT convention), not an absolute one — otherwise the same σ means
#      wildly different things depending on the overall scale of Λ. Scaling by norm(Λ)/norm(Ε)
#      makes `σ * norm(Λ) * Ε/norm(Ε)` have Frobenius norm ≈ σ * norm(Λ), i.e. σ=0.15 ≈ 15% relative noise.
Ε = randn(size(Λ))
Ε = 0.5 .* (Ε + Ε') # symmetrize the noise itself, not just Λ + noise
Λn = Λ + σ * (norm(Λ) / norm(Ε)) * Ε
Λn = 0.5 .* (Λn + Λn') # re-symmetrize in case of floating point round-off


504×504 Matrix{Float64}:
  1.62584    0.709279   0.710764  …  -0.180627  -0.264517  -0.264497
  0.709279   1.11765    0.781477     -0.17518   -0.257365  -0.257345
  0.710764   0.781477   1.12438      -0.175306  -0.25753   -0.257511
  1.70692    1.41371    1.15461      -0.268938  -0.350385  -0.35037
  1.70813    1.1534     1.41884      -0.26903   -0.350487  -0.350472
  0.696582   0.606852   0.613652  …  -0.171169  -0.251769  -0.25175
  1.02988    1.29842    1.04628      -0.261315  -0.341208  -0.341195
  0.529593   0.589127   0.542796     -0.166621  -0.245645  -0.245626
  0.917698   0.888887   0.862457     -0.254428  -0.332797  -0.332784
  0.480741   0.470955   0.466727     -0.162163  -0.239604  -0.239585
  ⋮                               ⋱                        
 -0.166615  -0.161613  -0.161725  …   1.44934    2.24387    2.38217
 -0.245239  -0.238697  -0.238842      2.04096    2.74515    2.83423
 -0.171032  -0.165877  -0.165994      2.0592     2.85053    2.72928
 -0.251516  -0.244764  

In [39]:

F, G, Σ = svd_on_modes(Λn,num_modes)

# svd_on_modes already returns G projected onto the boundary basis (fe.m rows),
# so it must not be passed through fe.down again -- just re-normalize and lift.
F = mean_zero_boundary(F) # Don't know if this is necessary
G = fe.up(mean_zero_boundary(G))

16129×255 Matrix{Float64}:
 -0.000504027  -0.049969   -0.0175856  0.0688998  …  -1.44371e-6  -1.28983e-6
 -0.000460607  -0.04853    -0.0172342  0.0664501     -2.22149e-6   1.01995e-5
  0.0           0.0         0.0        0.0            0.0          0.0
 -0.00045131   -0.0485785  -0.0172355  0.0664681     -2.12571e-6   1.44659e-6
 -0.000436736  -0.0683719  -0.0232797  0.0950938      1.04389e-6   9.12073e-6
  0.0           0.0         0.0        0.0        …   0.0          0.0
  0.0           0.0         0.0        0.0            0.0          0.0
 -0.000433813  -0.0683976  -0.0232809  0.0951049      1.15813e-6   1.38985e-6
  0.0           0.0         0.0        0.0            0.0          0.0
 -0.000447998  -0.047345   -0.0169484  0.0644487     -1.71306e-6  -1.42457e-5
  ⋮                                               ⋱               
  0.0           0.0         0.0        0.0        …   0.0          0.0
  0.0824999     0.0928738  -0.0628423  0.0362077     -1.73728e-7  -1.01683e-5
  0.0

In [40]:
fbm = FerriteBlockMode(F, G, fe)

FerriteBlockMode(ModularEIT.BlockLAssembler(9, [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0]), [1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0  …  1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0], sparse([1, 2, 3, 4, 5, 6, 7, 8, 9, 1  …  16129, 15868, 15872, 15874, 16122, 16123, 16126, 16127, 16128, 16129], [1, 1, 1, 1, 1, 1, 1, 1, 1, 2  …  16128, 16129, 16129, 16129, 16129, 16129, 16129, 16129, 16129, 16129], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 16129, 16129), sparse([1, 2, 3, 4, 5, 6, 7, 8, 9, 16130  …  16112, 16114, 16116, 16118, 16120, 16122, 16124, 16126, 16127, 16128], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  16130, 16130, 16130, 16130, 16130, 16130, 16130, 16130, 16130, 16130], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], 16130, 16130), [-0.05858491755663077 -3.501463612198256 … -5.227512572770059e-7 -4.6101523271586677e-7

In [41]:
mean(G[:,1])

-5.506717215575208e-20

In [42]:
#Either this
f, ∂f =  create_block_f∂f(fbm, fe; nmodes=25, block=true, gn=true, λ=1e-4)
prox_obj = create_prox_linesearch(f,∂f,ρ_obj)

(::ModularEIT.var"#prox#90"{Float64, ModularEIT.var"#prox#87#91"{ModularEIT.var"#53#54"{Int64, Bool, Float64, Nothing, Float64, FerriteBlockMode, FerriteFESpace{RefQuadrilateral}, Base.RefValue{Bool}, Base.RefValue{Float64}, Base.RefValue{Union{Nothing, Vector{Float64}}}}, ModularEIT.var"#57#58"{Int64, Bool, Float64, Bool, Float64, Nothing, Float64, FerriteBlockMode, FerriteFESpace{RefQuadrilateral}, ModularEIT.var"#53#54"{Int64, Bool, Float64, Nothing, Float64, FerriteBlockMode, FerriteFESpace{RefQuadrilateral}, Base.RefValue{Bool}, Base.RefValue{Float64}, Base.RefValue{Union{Nothing, Vector{Float64}}}}, Base.RefValue{Bool}, Base.RefValue{Union{Nothing, Vector{Float64}}}, Base.RefValue{Union{Nothing, Vector{Float64}}}}}, Bool, Nothing, Float64, Float64, Int64}) (generic function with 2 methods)

In [43]:
#Or this
f, ∂f =  create_block_f∂f(fbm, fe; nmodes=25, block=true, gn=false)
prox_obj = create_proximal_gradient_step(f,∂f,ρ_obj, fe.n_σ, maxiter= 1500)

#103 (generic function with 1 method)

In [44]:
σ₁ = ones(fe.n_σ)
# I wonder why I can't initialize it with an exp(randn(fe.n_σ)). It seems that this will not converge... or just get stuck at the beginning.

3969-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 ⋮
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0

In [ ]:
# NOTE: this was commented out, so σᵢ was never defined — the σ_img cell below would
# error with `UndefVarError: σᵢ`. Uncommented so the reconstruction actually runs.
σᵢ, err, _ = prox_obj(σ₁)

In [ ]:
σ_img = log.(reshape(evaluate_at_points(ph, fe.dh_σ, σᵢ), (64, 64)))

In [ ]:
σ_img = Gray.(((σ_img) .*0.5) .+0.5)

In [ ]:
#save("linesearch_25_unregularized.png", map(clamp01nan, σ_img))

In [ ]:
# cond_vec lives on the σ-space (fe.dh_σ), not the u-space (fe.dh) — must
# evaluate against fe.dh_σ to read the conductivity image back out.
cond_img = reshape(evaluate_at_points(ph, fe.dh_σ, cond_vec), (64, 64))
cond_img = Gray.(log.(cond_img) .*0.5 .+0.5)

In [ ]:
#save("original.png", map(clamp01nan, cond_img))

In [ ]:
Λ⁻¹